# Daisy Peri P — Calibration Curve Fitting

Peristaltic pumps are not perfectly linear across their full rpm range. This notebook measures the actual flow rate at multiple rpm points across the Peri P's full range (up to 150 rpm), then fits several mathematical models to the data to find which equation best describes the relationship.

Once the best-fit equation is found, you can use it to look up the accurate calibration value at any rpm.

---

## Models tested

- **Linear** — `flow = a × rpm + b` — assumes perfect proportionality
- **Quadratic** — `flow = a × rpm² + b × rpm + c` — allows for gentle curve
- **Power law** — `flow = a × rpm^b` — common model for pump behaviour

The best fit is selected by the highest R² value (closest to 1.0).

---

## Test points

The notebook runs the **Daisy Peri P** pump at 8 rpm points spread across the full range:
**20, 40, 60, 75, 90, 105, 120, 150 rpm**

Each run collects for 10 seconds. You measure the volume and enter it after each run.

## Import additional python libraries

The Daisy Python Library is already loaded into the Daisy Controller Software, all you have to do is import any external libraries you need. Here, we are using additional libraries to plot your recorded values and fit the resulting plot with three potential models.

In [1]:
%matplotlib widget
import time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.optimize import curve_fit
from scipy.stats import pearsonr

## Print instruments found
Print the instruments found to easily verify the index of the Daisy you will be using.

In [ ]:
print("Instruments found:")
for position, (code, inst) in enumerate(zip(self.my_interface.instruments_list[0], self.my_interface.groups[0].inst)):
    print(f"  [{position}]  {code} — {inst.name}")

## Set your instrument variable name
Change `INST_INDEX` to the position of the Daisy (Peri P) you want to use.

In [ ]:
# Set which instrument you are calibrating
INST_INDEX = 0   # 0 if first instrument or change to your pump's position

pump = self.my_interface.groups[0].inst[INST_INDEX]
print(f"Pump: {pump.name}")

## Helper functions

Here are some helper functions to assist in calculating what flow rate results in specific rpms and what models are the best fit.

In [ ]:
COLLECTION_TIME_S = 10  # Collection time is in seconds
PERI_P_MAX_RPM    = 150 # This is the max rpm for the Peri T pumps

# Calculating run parameters for Peri T to run at specific rpms
def rpm_to_speed(target_rpm, calibration_speed):
    """Convert a target rpm to the ml/min value to pass to pump.run()."""
    return (target_rpm / 60) * calibration_speed

def run_at_rpm(pump, target_rpm, duration_s=60):
    """Run the pump at target_rpm for duration_s seconds."""
    speed = rpm_to_speed(target_rpm, pump.calibration_speed)
    print(f"\nRunning at {target_rpm} rpm ({speed:.3f} ml/min)...")
    pump.run(speed, 999.0, wait=False)
    for remaining in range(duration_s, 0, -1):
        print(f"  {remaining}s remaining...", end='\r')
        time.sleep(1)
    pump.stop()
    print(f"\nPump stopped. Measure the volume collected.")

# Models for curve fitting
def model_linear(rpm, a, b):
    return a * rpm + b

def model_quadratic(rpm, a, b, c):
    return a * rpm**2 + b * rpm + c

def model_power(rpm, a, b):
    return a * np.power(rpm, b)

def r_squared(y_actual, y_predicted):
    ss_res = np.sum((y_actual - y_predicted) ** 2)
    ss_tot = np.sum((y_actual - np.mean(y_actual)) ** 2)
    return 1 - (ss_res / ss_tot)

print("Helper functions ready.")

## Prime your Peri P tubing

Before measuring, make sure the tubing is fully primed with no air gaps. Set `prime_volume` based on your **tubing size** and **length**.

**Peri P** uses smaller tubing than our other Peri pumps, you may only need 2-5 ml.

Re-run or increase `prime_volume` if your tubing is not fully primed.

In [ ]:
# prime_volume starting point for Peri P
# Adjust based on your tubing size and length
prime_volume = 2.0 # In ml

pump.run(pump.calibration_speed, prime_volume, wait=True)
print("Confirm liquid has primed all of the tubing.")

## **To stop the pump run:** 

Use the Jupyter stop button (■) in the toolbar (or press `I` twice to interrupt the kernel) to stop the `pump.run()` function above. Then run the next cell.

In [ ]:
pump.stop()

## Run your pump at each rpm and enter measured volumes

Run each cell below in order. After each run, tare your collection vessel, then enter the measured volume in the cell that follows before running the next rpm. You must run each cell with the entered volume to assign the variable to the collected volume. Water has a density of 1 gram / 1 ml. If you are weighing your solution and it does not have a density of 1 g/ml, calculate your volume before entering below.

**Each cell waits 2 seconds before starting — have your collection vessel ready.**

### Run at 20 rpm

In [ ]:
# ---- 20 rpm ----
print("Place your tared collection vessel under the outlet.")
print("Starting in 2 seconds...")
time.sleep(2)
run_at_rpm(pump, 20, COLLECTION_TIME_S)

In [ ]:
# Enter volume collected
vol_20rpm = 0.0

### Run at 40 rpm

In [ ]:
# ---- 40 rpm ----
print("Starting in 2 seconds...")
time.sleep(2)
run_at_rpm(pump, 40, COLLECTION_TIME_S)

In [ ]:
# Enter volume collected
vol_40rpm = 0.0

### Run at 60 rpm

In [ ]:
# ---- 60 rpm ----
print("Starting in 2 seconds...")
time.sleep(2)
run_at_rpm(pump, 60, COLLECTION_TIME_S)

In [ ]:
# Enter volume collected
vol_60rpm = 0.0

### Run at 75 rpm

In [ ]:
# ---- 75 rpm ----
print("Starting in 2 seconds...")
time.sleep(2)
run_at_rpm(pump, 75, COLLECTION_TIME_S)

In [ ]:
# Enter volume collected
vol_75rpm = 0.0

### Run at 90 rpm

In [ ]:
# ---- 90 rpm ----
print("Starting in 2 seconds...")
time.sleep(2)
run_at_rpm(pump, 90, COLLECTION_TIME_S)

In [ ]:
# Enter volume collected
vol_90rpm = 0.0

### Run at 105 rpm

In [ ]:
# ---- 105 rpm ----
print("Starting in 2 seconds...")
time.sleep(2)
run_at_rpm(pump, 105, COLLECTION_TIME_S)

In [ ]:
# Enter volume collected
vol_105rpm = 0.0

### Run at 120 rpm

In [ ]:
# ---- 120 rpm ----
print("Starting in 2 seconds...")
time.sleep(2)
run_at_rpm(pump, 120, COLLECTION_TIME_S)

In [ ]:
# Enter volume collected
vol_120rpm = 0.0

### Run at 150 rpm

In [ ]:
# ---- 150 rpm ----
print("Starting in 2 seconds...")
time.sleep(2)
run_at_rpm(pump, 150, COLLECTION_TIME_S)

In [ ]:
# Enter volume collected
vol_150rpm = 0.0

## Fit curves and plot results

Run this cell after entering all measured volumes. It fits three models to your data, prints the R² for each, identifies the best fit, and plots everything.

In [ ]:
# ---- Assemble data ----
rpms   = np.array([20,       40,       60,       75,       90,       105,       120,       150])
flows  = np.array([vol_20rpm, vol_40rpm, vol_60rpm, vol_75rpm, vol_90rpm, vol_105rpm, vol_120rpm, vol_150rpm]) / (COLLECTION_TIME_S / 60)


print("Measured data:")
print(f"  {'rpm':<8} {'Flow (ml/min)':<15}")
for r, f in zip(rpms, flows):
    print(f"  {r:<8} {f:<15.3f}")
print()

# ---- Fit models ----
rpm_smooth = np.linspace(rpms[0], rpms[-1], 300)
results = {}

# ---- Linear ---- 
try:
    popt, _ = curve_fit(model_linear, rpms, flows)
    pred = model_linear(rpms, *popt)
    r2 = r_squared(flows, pred)
    results["Linear"] = {"params": popt, "r2": r2, "func": model_linear,
                          "label": f"Linear: flow = {popt[0]:.4f}×rpm + {popt[1]:.4f}"}
except Exception as e:
    print(f"Linear fit failed: {e}")

# ---- Quadratic ---- 
try:
    popt, _ = curve_fit(model_quadratic, rpms, flows)
    pred = model_quadratic(rpms, *popt)
    r2 = r_squared(flows, pred)
    results["Quadratic"] = {"params": popt, "r2": r2, "func": model_quadratic,
                             "label": f"Quadratic: flow = {popt[0]:.5f}×rpm² + {popt[1]:.4f}×rpm + {popt[2]:.4f}"}
except Exception as e:
    print(f"Quadratic fit failed: {e}")

# ---- Power law ---- 
try:
    popt, _ = curve_fit(model_power, rpms, flows, p0=[0.01, 1.0], maxfev=10000)
    pred = model_power(rpms, *popt)
    r2 = r_squared(flows, pred)
    results["Power law"] = {"params": popt, "r2": r2, "func": model_power,
                             "label": f"Power law: flow = {popt[0]:.5f}×rpm^{popt[1]:.4f}"}
except Exception as e:
    print(f"Power law fit failed: {e}")

# ---- Print R² comparison ----
print("Model fit comparison (R² closer to 1.0 is better):")
print(f"  {'Model':<12} {'R²':<10} {'Equation'}")
print("-" * 70)
best_model = max(results, key=lambda k: results[k]["r2"])
for name, data in results.items():
    marker = " ← best fit" if name == best_model else ""
    print(f"  {name:<12} {data['r2']:.6f}  {data['label']}{marker}")
print()

# ---- Plot ----
colors = {"Linear": "cyan", "Quadratic": "orange", "Power law": "magenta"}

fig, ax = plt.subplots(figsize=(10, 5))
ax.set_facecolor('black')
fig.patch.set_facecolor('black')
ax.tick_params(colors='white')
ax.set_xlabel('rpm', color='white')
ax.set_ylabel('Flow Rate (ml/min)', color='white')
ax.set_title('Peri P — rpm vs Flow Rate with Curve Fits', color='white')
for spine in ax.spines.values():
    spine.set_edgecolor('white')
ax.grid(True, alpha=0.3, color='white')

# Measured data points
ax.scatter(rpms, flows, color='lime', s=60, zorder=5, label='Measured')

# Fit curves
for name, data in results.items():
    y_smooth = data["func"](rpm_smooth, *data["params"])
    lw = 2 if name == best_model else 1
    ls = '-' if name == best_model else '--'
    ax.plot(rpm_smooth, y_smooth, color=colors[name], linewidth=lw,
            linestyle=ls, label=f"{name} (R²={data['r2']:.4f})")

ax.legend(facecolor='black', labelcolor='white', fontsize=8)
plt.tight_layout()
plt.show()

print(f"\nBest fit: {best_model}")
print(f"Equation: {results[best_model]['label']}")

## Look up calibration at any rpm

Use the best-fit equation to find the accurate flow rate (and therefore calibration value) at any rpm in your operating range.

In [ ]:
# Enter any rpm to calculate the corresponding flow rate
QUERY_RPM = 60.0

best = results[best_model]
predicted_flow = best["func"](QUERY_RPM, *best["params"])

# Calibration value = predicted flow at query_rpm, scaled to 60 rpm equivalent
cal_equivalent = predicted_flow / (QUERY_RPM / 60)

print(f"At {QUERY_RPM} rpm:")
print(f"  Predicted flow rate:          {predicted_flow:.3f} ml/min")
print(f"  Calibration equivalent @ 60 rpm: {cal_equivalent:.3f} ml/min")
print()
print(f"To run at {QUERY_RPM} rpm with your pump:")
print(f"  pump.calibrate({cal_equivalent:.3f})")
print(f"  pump.run({cal_equivalent:.3f} * ({query_rpm} / 60), volume, wait=True)")
print(f"  # i.e. pump.run({predicted_flow:.3f}, volume, wait=True)")

## Print full calibration table for your Peri P

Prints the predicted flow rate and calibration equivalent at every 10 rpm across the full Peri P range.

In [ ]:
best = results[best_model]

print(f"Peri P calibration table — {best_model} fit")
print("-" * 50)
print(f"  {'rpm':<8} {'Flow (ml/min)':<20} {'Speed arg for pump.run()'}")
print("-" * 50)
for rpm in range(20, PERI_P_MAX_RPM + 1, 10):
    flow = best["func"](rpm, *best["params"])
    print(f"  {rpm:<8} {flow:<20.3f} pump.run({flow:.3f}, volume, wait=True)")
print("-" * 50)

## Close the connection to your Daisy instrument(s)

In [ ]:
lab.close()
print("Connection closed.")